# Phase 2 — Data understanding

Thin notebook that orchestrates the modules in `seercast.data` to:

1. Load the raw M5 CSVs from `data/raw/`.
2. Melt sales wide → long, filter to `store_id == 'CA_1'`, join calendar + sell_prices.
3. Run validation (schema, missing dates, duplicates, negative sales, missing-price rate).
4. Persist the result to `data/interim/m5_base_ca1.parquet`.

All real logic lives in `src/seercast/data/`. This notebook is a runnable demo and a place to look at the data, not a place to write logic.

**Prereq:** the raw competition CSVs (`calendar.csv`, `sales_train_validation.csv`, `sales_train_evaluation.csv`, `sell_prices.csv`, `sample_submission.csv`) must be in `data/raw/`.

In [ ]:
import sys
from pathlib import Path

# Make `src/` importable when the notebook is run before `pip install -e .`.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd

from seercast.config import ARTIFACTS, DEFAULT_STORE_ID, ensure_dirs
from seercast.data import build_base_table, load_m5_raw, validate_base_table

ensure_dirs()
REPO_ROOT

## 1. Load raw frames

In [ ]:
raw = load_m5_raw(REPO_ROOT / "data")

print("calendar:", raw.calendar.shape)
print("sales   :", raw.sales.shape)
print("prices  :", raw.prices.shape)
raw.calendar.head()

In [ ]:
raw.sales.iloc[:3, :10]

## 2. Build the joined base table for CA_1

In [ ]:
base = build_base_table(raw=raw, store_ids=(DEFAULT_STORE_ID,))
print("base shape:", base.shape)
base.head()

## 3. Validate

In [ ]:
report = validate_base_table(base)
print(report.summary())

## 4. Quick eyeball: daily total sales for CA_1

A sanity check that the join lines up: this should look like a noisy upward-trending series with weekly seasonality.

In [ ]:
import matplotlib.pyplot as plt

daily = base.groupby("date", as_index=False)["sales"].sum()
ax = daily.plot(x="date", y="sales", figsize=(12, 3), legend=False)
ax.set_title(f"Daily total unit sales — {DEFAULT_STORE_ID}")
ax.set_ylabel("units")
plt.tight_layout()
plt.show()

## 5. Persist the base table

In [ ]:
out = ARTIFACTS.base_table_ca1
out.parent.mkdir(parents=True, exist_ok=True)
base.to_parquet(out, index=False)
print("wrote:", out, "(", out.stat().st_size / 1e6, "MB )")

**Next:** Phase 3 — baselines and rolling-origin backtesting.